In [1]:
#Date 24th 24 Jun — Calculate total unique invoices per customer Goal
import pandas as pd
df = pd.read_csv("C:/Users/yash6/Internship 1/cleaned_retail_data_updated.csv", encoding='latin-1')
print(df.head())
print(df.shape)


   InvoiceNo StockCode                          Description  Quantity  \
0     536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1     536365     71053                  WHITE METAL LANTERN         6   
2     536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3     536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4     536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

        InvoiceDate  UnitPrice  CustomerID         Country TransactionMonth  \
0  01-12-2010 08:26       2.55       17850  United Kingdom       01-12-2010   
1  01-12-2010 08:26       3.39       17850  United Kingdom       01-12-2010   
2  01-12-2010 08:26       2.75       17850  United Kingdom       01-12-2010   
3  01-12-2010 08:26       3.39       17850  United Kingdom       01-12-2010   
4  01-12-2010 08:26       3.39       17850  United Kingdom       01-12-2010   

  CohortMonth  CohortIndex  
0  01-12-2010          0.0  
1  01-12-2010          0.0  

In [2]:
#Remove rows where CustomerID is missing
df = df.dropna(subset=["CustomerID"])

In [3]:
# Convert CustomerID into integer
df["CustomerID"] = df["CustomerID"].astype(int)

In [4]:
# Count unique customer
customer_frequency = (
    df.groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .reset_index()
)


In [5]:
# Rename of column
customer_frequency.columns = ["CustomerID", "PurchaseFrequency"]

In [6]:
import os
# Create outputs folder
output_folder = "outputs"
os.makedirs(output_folder, exist_ok=True)

In [7]:
# 7. Save the file in the folder
file_path = os.path.join(
    output_folder,
    "customer_purchase_frequency.csv"
)
customer_frequency.to_csv(file_path, index=False)


In [8]:
# 8. Show output
print("Successfully saved!")
print("File location:", os.path.abspath(file_path))
print("Total customers:", len(customer_frequency))

display(customer_frequency.head())

Successfully saved!
File location: C:\Users\yash6\Internship 1\outputs\customer_purchase_frequency.csv
Total customers: 4338


,CustomerID,PurchaseFrequency
0,12346,1
1,12347,7
2,12348,4
3,12349,1
4,12350,1


In [9]:
#25 Jun — Calculate average purchase frequency per cohort segment

In [11]:
# Get one cohort month per customer
customer_cohort = (
    df.groupby("CustomerID")["CohortMonth"]
    .min()
    .reset_index()
)

In [12]:
# Merge purchase frequency with cohort information
frequency_with_cohort = customer_frequency.merge(
    customer_cohort,
    on="CustomerID",
    how="left"
)

In [13]:
# Calculate average purchase frequency by cohort
cohort_frequency = (
    frequency_with_cohort.groupby("CohortMonth")["PurchaseFrequency"]
    .agg(
        AveragePurchaseFrequency="mean",
        TotalCustomers="count",
        MedianPurchaseFrequency="median"
    )
    .reset_index()
)


In [14]:
# Round average values
cohort_frequency["AveragePurchaseFrequency"] = (
    cohort_frequency["AveragePurchaseFrequency"].round(2)
)

print(cohort_frequency.head())

  CohortMonth  AveragePurchaseFrequency  TotalCustomers  \
0  01-01-2011                      6.83             178   
1  01-02-2011                      5.78             190   
2  01-03-2011                      4.67             234   
3  01-04-2011                      5.09             232   
4  01-05-2011                      4.74             253   

   MedianPurchaseFrequency  
0                      5.0  
1                      4.0  
2                      3.5  
3                      4.0  
4                      4.0  


In [16]:
import os

# Your real outputs folder
output_folder = r"C:\Users\yash6\Internship 1\outputs"

# Create it automatically if missing
os.makedirs(output_folder, exist_ok=True)

# Correct file location
output_file = os.path.join(
    output_folder,
    "cohort_purchase_frequency.csv"
)
